In [70]:
import numpy as np
import pandas as pd
##import matplotlib.pyplot as plt
# import seaborn as sb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

from sklearn import metrics


import warnings
warnings.filterwarnings('ignore')

In [71]:
data_path = '/home/user171125/Documents/AI_PROJECTS/Rule_based_system/Model_training/Dataset/realistic_truck_dispatch_dataset_1000.csv'
try:
    df = pd.read_csv(data_path)
    print(f"Dataset loaded with shape: {df.shape}")
    display(df.head())
except Exception as e:
    print("Error loading dataset. Make sure you run `generate_dataset.py` first.")
    print(e)


Dataset loaded with shape: (1000, 26)


,id,driver_id,load_id,bid_id,driver_rating,completion_rate,acceptance_rate,total_jobs,load_weight,distance_km,...,past_route_success,bid_price,price_deviation,response_time,load_type,driver_score,price_per_km,outcome,label,created_at
0,0351f7ed-3eaa-46a8-aafc-1f7778fa3846,218bee8a-2616-4e65-8f53-66622f058a51,bec786fc-c4f4-4c28-83c6-5330ea3f84f2,5e657c14-812a-4df3-99c3-938a52b304ec,3.88,0.46,0.38,72,7.89,430.34,...,0.35,1156.21,0.05,227.18,flatbed,0.1743,2.6805,rejected,0,2025-06-05T09:04:56
1,189216f3-94b1-4d57-88a4-f0ac0e071f29,4cb23c0a-094a-4203-b2f4-5f98dd05afe7,43f10cfa-22f8-40aa-89cf-016fe04223fa,3b7d96cf-f862-49f3-a683-bcaa91c926cb,3.66,0.45,0.45,78,5.32,156.00,...,0.40,484.24,0.03,259.53,flatbed,0.2027,3.0843,completed,1,2025-07-30T09:04:56
2,3cbf1580-bac9-4aea-ad3a-68824e3c1b7d,03985783-1b65-4184-8f24-a5a3ea4f8dd4,6e331928-3ac1-4d57-9e54-afa714f5b099,05bbd7a3-1be6-4fcb-8e31-4b586bd784a4,4.26,0.51,0.56,78,20.62,3652.92,...,0.62,9052.34,0.12,138.32,dry_van,0.2830,2.4774,accepted,1,2026-02-05T09:04:56
3,fc30f7e7-ec92-4202-a8da-0bc4d11cfa44,f90094c5-de8f-40af-82dc-f6bb189d75ed,c6ff00d9-8553-4d48-b4ab-54711b5fbd52,f3b51417-6379-48a9-9e80-17cda1d8c75e,4.09,0.47,0.47,62,23.42,103.66,...,0.46,315.84,-0.22,239.69,reefer,0.2192,3.0178,rejected,0,2025-08-08T09:04:56
4,6b17f953-7711-4ee7-8b53-5a9b7a915156,ed970389-94c8-46e9-ac06-50bbdfbba1ab,9e76c64c-9990-4906-afbe-66dc1693dfbf,05068056-4723-41cd-8faf-53169d39887d,4.70,0.61,0.64,110,13.56,736.46,...,0.45,1731.03,0.04,162.62,dry_van,0.3893,2.3473,completed,1,2025-10-30T09:04:56


In [72]:
df_sample =df.copy()
df_sample = df_sample.drop(columns=['load_id', 'driver_id','id','bid_id','created_at'])
df_sample.head(10)

,driver_rating,completion_rate,acceptance_rate,total_jobs,load_weight,distance_km,pickup_lat,pickup_lng,drop_lat,drop_lng,...,driver_to_pickup_distance,past_route_success,bid_price,price_deviation,response_time,load_type,driver_score,price_per_km,outcome,label
0,3.88,0.46,0.38,72,7.89,430.34,45.032342,-75.521804,46.911564,-70.652217,...,1.46,0.35,1156.21,0.05,227.18,flatbed,0.1743,2.6805,rejected,0
1,3.66,0.45,0.45,78,5.32,156.00,44.922838,-76.811896,45.813298,-75.268644,...,1.00,0.40,484.24,0.03,259.53,flatbed,0.2027,3.0843,completed,1
2,4.26,0.51,0.56,78,20.62,3652.92,44.131317,-63.971114,53.384876,-112.912383,...,81.25,0.62,9052.34,0.12,138.32,dry_van,0.2830,2.4774,accepted,1
3,4.09,0.47,0.47,62,23.42,103.66,47.155341,-70.956163,46.224399,-71.028235,...,1.00,0.46,315.84,-0.22,239.69,reefer,0.2192,3.0178,rejected,0
4,4.70,0.61,0.64,110,13.56,736.46,50.070607,-96.501397,52.223191,-106.497018,...,14.89,0.45,1731.03,0.04,162.62,dry_van,0.3893,2.3473,completed,1
5,3.68,0.49,0.41,95,30.00,694.04,46.712702,-71.216516,43.643618,-78.933370,...,37.40,0.42,1510.51,-0.22,173.36,flatbed,0.2018,2.1733,rejected,0
6,4.49,0.56,0.55,143,15.45,1559.04,49.904935,-97.796798,43.929585,-79.145881,...,124.30,0.27,4117.54,0.14,99.57,dry_van,0.3041,2.6394,completed,1
7,3.57,0.48,0.36,115,21.00,2747.25,43.912634,-79.240264,53.819738,-114.174123,...,84.37,0.38,9050.64,0.03,76.44,flatbed,0.1701,3.2932,rejected,0
8,4.39,0.54,0.63,624,12.74,2655.36,50.579174,-114.085808,44.123309,-79.787201,...,146.95,0.59,7771.04,0.13,141.09,flatbed,0.3431,2.9254,completed,1
9,4.77,0.58,0.58,56,12.37,3764.85,52.996316,-113.563983,44.768357,-62.409813,...,138.49,0.77,8721.61,-0.19,44.94,reefer,0.3403,2.3160,accepted,1


In [73]:
df_sample.isnull().sum()

driver_rating                0
completion_rate              0
acceptance_rate              0
total_jobs                   0
load_weight                  0
distance_km                  0
pickup_lat                   0
pickup_lng                   0
drop_lat                     0
drop_lng                     0
urgency                      0
driver_to_pickup_distance    0
past_route_success           0
bid_price                    0
price_deviation              0
response_time                0
load_type                    0
driver_score                 0
price_per_km                 0
outcome                      0
label                        0
dtype: int64

In [74]:
df_sample.describe()

,driver_rating,completion_rate,acceptance_rate,total_jobs,load_weight,distance_km,pickup_lat,pickup_lng,drop_lat,drop_lng,urgency,driver_to_pickup_distance,past_route_success,bid_price,price_deviation,response_time,driver_score,price_per_km,label
count,1000.00000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,4.12442,0.540160,0.495590,119.85800,14.024310,1719.515100,48.342766,-92.430446,48.176530,-90.954260,1.768000,80.743050,0.488090,4434.58425,0.007510,183.809350,0.271300,2.769870,0.588000
std,0.41322,0.074371,0.093248,89.76372,7.611312,1257.357828,3.234996,20.095506,3.275589,20.197802,0.778963,61.099981,0.157514,3443.63809,0.146934,79.993432,0.077945,0.926831,0.492441
min,2.86000,0.450000,0.350000,14.00000,1.000000,25.000000,42.824761,-124.206404,42.668050,-124.738869,1.000000,1.000000,0.000000,200.00000,-0.280000,20.000000,0.157500,1.350000,0.000000
25%,3.84000,0.470000,0.430000,64.00000,8.130000,527.697500,45.414267,-113.329746,45.239381,-113.138956,1.000000,25.850000,0.380000,1376.83750,-0.090000,129.032500,0.211975,2.232650,0.000000
50%,4.13000,0.530000,0.490000,98.00000,12.245000,1616.075000,48.818570,-96.582369,47.248107,-80.095085,2.000000,66.985000,0.490000,3868.72000,0.010000,181.605000,0.261150,2.590650,1.000000
75%,4.41000,0.590000,0.560000,149.25000,19.270000,2843.252500,51.092657,-73.784120,51.067713,-73.226118,2.000000,135.047500,0.590000,7052.01750,0.110000,239.507500,0.319600,3.050550,1.000000
max,5.00000,0.780000,0.810000,1200.00000,30.000000,4200.000000,54.412380,-62.294550,54.280837,-62.341225,3.000000,180.000000,1.000000,15532.83000,0.350000,439.440000,0.523100,7.692300,1.000000


In [75]:
df_sample.head(5)    

,driver_rating,completion_rate,acceptance_rate,total_jobs,load_weight,distance_km,pickup_lat,pickup_lng,drop_lat,drop_lng,...,driver_to_pickup_distance,past_route_success,bid_price,price_deviation,response_time,load_type,driver_score,price_per_km,outcome,label
0,3.88,0.46,0.38,72,7.89,430.34,45.032342,-75.521804,46.911564,-70.652217,...,1.46,0.35,1156.21,0.05,227.18,flatbed,0.1743,2.6805,rejected,0
1,3.66,0.45,0.45,78,5.32,156.00,44.922838,-76.811896,45.813298,-75.268644,...,1.00,0.40,484.24,0.03,259.53,flatbed,0.2027,3.0843,completed,1
2,4.26,0.51,0.56,78,20.62,3652.92,44.131317,-63.971114,53.384876,-112.912383,...,81.25,0.62,9052.34,0.12,138.32,dry_van,0.2830,2.4774,accepted,1
3,4.09,0.47,0.47,62,23.42,103.66,47.155341,-70.956163,46.224399,-71.028235,...,1.00,0.46,315.84,-0.22,239.69,reefer,0.2192,3.0178,rejected,0
4,4.70,0.61,0.64,110,13.56,736.46,50.070607,-96.501397,52.223191,-106.497018,...,14.89,0.45,1731.03,0.04,162.62,dry_van,0.3893,2.3473,completed,1


In [76]:
df_sample = pd.get_dummies(df_sample, columns=["load_type"])


In [77]:
df_sample.head(5)


,driver_rating,completion_rate,acceptance_rate,total_jobs,load_weight,distance_km,pickup_lat,pickup_lng,drop_lat,drop_lng,...,bid_price,price_deviation,response_time,driver_score,price_per_km,outcome,label,load_type_dry_van,load_type_flatbed,load_type_reefer
0,3.88,0.46,0.38,72,7.89,430.34,45.032342,-75.521804,46.911564,-70.652217,...,1156.21,0.05,227.18,0.1743,2.6805,rejected,0,False,True,False
1,3.66,0.45,0.45,78,5.32,156.00,44.922838,-76.811896,45.813298,-75.268644,...,484.24,0.03,259.53,0.2027,3.0843,completed,1,False,True,False
2,4.26,0.51,0.56,78,20.62,3652.92,44.131317,-63.971114,53.384876,-112.912383,...,9052.34,0.12,138.32,0.2830,2.4774,accepted,1,True,False,False
3,4.09,0.47,0.47,62,23.42,103.66,47.155341,-70.956163,46.224399,-71.028235,...,315.84,-0.22,239.69,0.2192,3.0178,rejected,0,False,False,True
4,4.70,0.61,0.64,110,13.56,736.46,50.070607,-96.501397,52.223191,-106.497018,...,1731.03,0.04,162.62,0.3893,2.3473,completed,1,True,False,False


In [78]:
cols = ["load_type_dry_van", "load_type_flatbed", "load_type_reefer"]
df_sample[cols] = df_sample[cols].astype(int)

In [79]:
df_sample.head(5)

,driver_rating,completion_rate,acceptance_rate,total_jobs,load_weight,distance_km,pickup_lat,pickup_lng,drop_lat,drop_lng,...,bid_price,price_deviation,response_time,driver_score,price_per_km,outcome,label,load_type_dry_van,load_type_flatbed,load_type_reefer
0,3.88,0.46,0.38,72,7.89,430.34,45.032342,-75.521804,46.911564,-70.652217,...,1156.21,0.05,227.18,0.1743,2.6805,rejected,0,0,1,0
1,3.66,0.45,0.45,78,5.32,156.00,44.922838,-76.811896,45.813298,-75.268644,...,484.24,0.03,259.53,0.2027,3.0843,completed,1,0,1,0
2,4.26,0.51,0.56,78,20.62,3652.92,44.131317,-63.971114,53.384876,-112.912383,...,9052.34,0.12,138.32,0.2830,2.4774,accepted,1,1,0,0
3,4.09,0.47,0.47,62,23.42,103.66,47.155341,-70.956163,46.224399,-71.028235,...,315.84,-0.22,239.69,0.2192,3.0178,rejected,0,0,0,1
4,4.70,0.61,0.64,110,13.56,736.46,50.070607,-96.501397,52.223191,-106.497018,...,1731.03,0.04,162.62,0.3893,2.3473,completed,1,1,0,0


In [80]:
df_sample["driver_score"] = df_sample["completion_rate"] * df_sample["acceptance_rate"]
df_sample.head(5)
df_sample['label'].sum()

np.int64(588)

In [81]:
drop_cols = [
    "outcome",   # leakage
    "label",     # target (separate y)
    
    # IDs (if present)
    "id",
    "driver_id",
    "load_id",
    "bid_id",

    # redundant geo (since distance exists)
    "pickup_lat",
    "pickup_lng",
    "drop_lat",
    "drop_lng"
]
x = df_sample.drop(columns=drop_cols, errors="ignore")
y = df_sample["label"]

In [82]:
y.head(5)

0    0
1    1
2    1
3    0
4    1
Name: label, dtype: int64

In [83]:
x.head(5)

,driver_rating,completion_rate,acceptance_rate,total_jobs,load_weight,distance_km,urgency,driver_to_pickup_distance,past_route_success,bid_price,price_deviation,response_time,driver_score,price_per_km,load_type_dry_van,load_type_flatbed,load_type_reefer
0,3.88,0.46,0.38,72,7.89,430.34,1,1.46,0.35,1156.21,0.05,227.18,0.1748,2.6805,0,1,0
1,3.66,0.45,0.45,78,5.32,156.00,3,1.00,0.40,484.24,0.03,259.53,0.2025,3.0843,0,1,0
2,4.26,0.51,0.56,78,20.62,3652.92,2,81.25,0.62,9052.34,0.12,138.32,0.2856,2.4774,1,0,0
3,4.09,0.47,0.47,62,23.42,103.66,2,1.00,0.46,315.84,-0.22,239.69,0.2209,3.0178,0,0,1
4,4.70,0.61,0.64,110,13.56,736.46,3,14.89,0.45,1731.03,0.04,162.62,0.3904,2.3473,1,0,0


In [84]:
df_sample.isnull().sum()


driver_rating                0
completion_rate              0
acceptance_rate              0
total_jobs                   0
load_weight                  0
distance_km                  0
pickup_lat                   0
pickup_lng                   0
drop_lat                     0
drop_lng                     0
urgency                      0
driver_to_pickup_distance    0
past_route_success           0
bid_price                    0
price_deviation              0
response_time                0
driver_score                 0
price_per_km                 0
outcome                      0
label                        0
load_type_dry_van            0
load_type_flatbed            0
load_type_reefer             0
dtype: int64

In [85]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [86]:
y_test.head(5)

521    0
737    1
740    0
660    0
411    1
Name: label, dtype: int64

In [87]:
! pip uninstall xgboost -y
! pip install xgboost==2.0.3

Found existing installation: xgboost 2.0.3
Uninstalling xgboost-2.0.3:
  Successfully uninstalled xgboost-2.0.3
  Using cached xgboost-2.0.3-py3-none-manylinux2014_x86_64.whl.metadata (2.0 kB)
Using cached xgboost-2.0.3-py3-none-manylinux2014_x86_64.whl (297.1 MB)


In [88]:
# RECOMMENDED
from xgboost import XGBClassifier
xgb_model = XGBClassifier(      # objective="rank:ndcg",xgbranker ----> new model
    n_estimators=300,            # ✅ more rounds
    max_depth=4,
    learning_rate=0.03,          # ✅ slower learning
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,               # ✅ L1 regularization
    reg_lambda=1.0,              # ✅ L2 regularization
    scale_pos_weight=0.7,        # ✅ handle class imbalance (412/588)
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=True,
    early_stopping_rounds=20)    # ✅ auto-stop when plateau

[0]	validation_0-logloss:0.69483
[1]	validation_0-logloss:0.68850
[2]	validation_0-logloss:0.68370
[3]	validation_0-logloss:0.67779
[4]	validation_0-logloss:0.67256
[5]	validation_0-logloss:0.66904
[6]	validation_0-logloss:0.66690
[7]	validation_0-logloss:0.66295
[8]	validation_0-logloss:0.65875
[9]	validation_0-logloss:0.65562
[10]	validation_0-logloss:0.65233
[11]	validation_0-logloss:0.64882
[12]	validation_0-logloss:0.64536
[13]	validation_0-logloss:0.64218
[14]	validation_0-logloss:0.63919
[15]	validation_0-logloss:0.63610
[16]	validation_0-logloss:0.63331
[17]	validation_0-logloss:0.63137
[18]	validation_0-logloss:0.62882
[19]	validation_0-logloss:0.62587
[20]	validation_0-logloss:0.62350
[21]	validation_0-logloss:0.62099
[22]	validation_0-logloss:0.61925
[23]	validation_0-logloss:0.61773
[24]	validation_0-logloss:0.61520
[25]	validation_0-logloss:0.61356
[26]	validation_0-logloss:0.61221
[27]	validation_0-logloss:0.61096
[28]	validation_0-logloss:0.60895
[29]	validation_0-loglos

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [94]:
from sklearn.metrics import roc_auc_score

y_prob = xgb_model.predict_proba(X_test)[:,1]
print("AUC:", roc_auc_score(y_test, y_prob))

AUC: 0.7414518390680617


In [95]:
from sklearn.metrics import accuracy_score

y_pred = xgb_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.685


In [96]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[48 25]
 [38 89]]


In [92]:
# import pickle
# import os

# # Create directory if it doesn't exist
# model_dir = "/home/user171125/Documents/AI_PROJECTS/Rule_based_system/Model_training/Model"
# model_path = os.path.join(model_dir, "xgb_model.pkl")

# # Dump model
# with open(model_path, "wb") as f:
#     pickle.dump(xgb_model, f)

# # Also save feature names for inference
# feature_path = os.path.join(model_dir, "feature_names.pkl")
# with open(feature_path, "wb") as f:
#     pickle.dump(list(X_train.columns), f)

# print(f"✅ Model saved to: {model_path}")
# print(f"✅ Features saved to: {feature_path}")
# print(f"📦 Model size: {os.path.getsize(model_path) / 1024:.1f} KB")


In [93]:
import pickle

with open("/home/user171125/Documents/AI_PROJECTS/Rule_based_system/Model_training/Model/feature_names.pkl", "rb") as f:
    features = pickle.load(f)
print(f"Total features: {len(features)}")
print(features)


Total features: 17
['driver_rating', 'completion_rate', 'acceptance_rate', 'total_jobs', 'load_weight', 'distance_km', 'urgency', 'driver_to_pickup_distance', 'past_route_success', 'bid_price', 'price_deviation', 'response_time', 'driver_score', 'price_per_km', 'load_type_dry_van', 'load_type_flatbed', 'load_type_reefer']
